In [ ]:
%load_ext autoreload
%autoreload 2
# automatically pick up changes made in src file

# LEDGAR: supervised models (logistic regression + Complement Naive Bayes)
Loads the features saved by `preprocess.ipynb` (run that once first), then trains, evaluates and compares both models.

**Flow**
1. Imports
2. Load saved features + labels
3. Evaluator (scores every model the same way)
4. Model 1: multiclass softmax logistic regression
5. Model 2: Complement Naive Bayes (alpha / norm tuning)
6. Compare the models (validation)
7. Error analysis (validation)
8. Save the trained models
9. Test (ONCE, at the very end)

`src/models.py` only TRAINS. All validating and testing happens through `src/sup_eval_metrics.py`.

## 1. Imports
All src files are imported here, once.

In [ ]:
from pathlib import Path
import sys; sys.path.append('..')     # lets the notebook find the src folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
import joblib

from src.models import MultiClassLogisticRegression, ComplementNaiveBayes   # model classes (train only)
from src.sup_eval_metrics import SupervisedEvaluator                       # scores any trained model
from src.classification_error_analysis import ErrorAnalysis                # error analysis class

## 2. Load the saved features (made by `preprocess.ipynb`)
No cleaning, lemmatising or TF-IDF here - everything is read from disk in seconds.
- `x_train`, `x_val`, `x_test`: TF-IDF matrices (sparse)
- `vectorizer` / `feature_names`: the fitted TF-IDF and its column (word) names
- `proc_train`, `proc_val`, `proc_test`: the lemmatised clauses with labels

In [ ]:
processed = Path("../data/processed")
features = Path("../data/features")

# TF-IDF matrices
x_train = sparse.load_npz(features / "x_train.npz")
x_val = sparse.load_npz(features / "x_val.npz")
x_test = sparse.load_npz(features / "x_test.npz")

# fitted vectoriser and its column (word) names
vectorizer = joblib.load(features / "vectorizer.joblib")
feature_names = vectorizer.get_feature_names_out()

# lemmatised clauses (text + labels)
proc_train = pd.read_parquet(processed / "proc_train.parquet")
proc_val = pd.read_parquet(processed / "proc_validation.parquet")
proc_test = pd.read_parquet(processed / "proc_test.parquet")

print("x_train:", x_train.shape, "| x_val:", x_val.shape, "| x_test:", x_test.shape)

Labels
- y = the label column of each split: y_train, y_val, y_test
- `label_names`: lookup from label number to clause name, e.g. 47 → "Governing Laws"

In [ ]:
y_train = proc_train.label
y_val = proc_val.label
y_test = proc_test.label

# pairs each number with its name, then makes a lookup dictionary
# zip lines up the two columns side by side
label_names = dict(zip(proc_train["label"], proc_train["label_name"]))

## 3. Evaluator
Stores the data splits ONCE. Every model is then scored the same way:
- `evaluate(model)`: train + validation + gap - compare train with validation to check for overfitting
- `validate(model)`: score on validation - unseen data, used to choose settings
- `compare({...})`: side-by-side table of several models
- `eval_test(model)`: final exam - run ONCE at the very end

Bias / variance: training error high → bias (underfitting); gap (val error − train error) big → variance (overfitting).

In [ ]:
ev = SupervisedEvaluator(x_train, y_train, x_val, y_val, x_test, y_test)   # x and y in pairs: train, val, test

## 4. Model 1: multiclass softmax logistic regression
- `train()`: learn the weights (train data only)
- the trained model is kept in `lr` and passed to the evaluator - no retraining between validation and test

In [ ]:
lr = MultiClassLogisticRegression(C=30.0, class_weight='balanced').train(x_train, y_train)  # learn weights

# the model guesses train + validation labels, and they're marked
val_preds_lr, scores_lr = ev.evaluate(lr.model)

## 5. Model 2: Complement Naive Bayes
Since the model initially started with underfitting (training error ~20%), test different alpha levels.
Lower alpha = less smoothing = less bias, more variance. Pick the best **validation** macro F1.

In [ ]:
# since the model initially started with underfitting, test different alpha levels (norm = False and True):
rows = []
for norm in [False, True]:
    for a in [1.0, 0.3, 0.1, 0.03, 0.01]:
        cnb = ComplementNaiveBayes(alpha=a, norm=norm).train(x_train, y_train)
        _, s = ev.evaluate(cnb.model)
        rows.append({"alpha": a, "norm": norm,
                     "train macro F1": s["train"]["macro_f1"],
                     "val macro F1": s["validation"]["macro_f1"],
                     "gap": s["gap"]})

alpha_results = pd.DataFrame(rows).round(4)
alpha_results

Results:
- validation macro F1 rose from ~0.70 (alpha = 1) to ~0.72, and plateaued below alpha = 0.1
- norm made no meaningful difference (≤ 0.3 pts)
- training error stayed ~16% → the remaining gap to logistic regression is model bias, not tuning
- **chosen: alpha = 0.1, norm = False** (simplest setting at the plateau)

In [ ]:
# final Naive Bayes, with the chosen settings
nb = ComplementNaiveBayes(alpha=0.1, norm=False).train(x_train, y_train)
val_preds_nb, scores_nb = ev.evaluate(nb.model)

## 6. Compare the models (validation only)

In [ ]:
ev.compare({"Logistic regression (C=30, balanced)": lr.model,
            "Complement NB (alpha=0.1)": nb.model})

## 7. Error analysis (validation set)
Uses `src/classification_error_analysis.py` → `ErrorAnalysis` class. Validation shows authentic weaknesses on unseen data.
1. Per-class scores, worst F1 first: which clause types are weak?
2. TP / FP / FN / TN per clause type
3. Top confused pairs: which types get mixed up with which?
4. One weak class in detail: where did its clauses go?
5. Read the actual wrong clauses
6. Confusion matrix picture (overview)
7. Group the mistakes into 3-4 reasons and count them (e.g. overlapping labels / too rare / long or merged clauses)

### 7a. Logistic regression

In [ ]:
ea = ErrorAnalysis(y_true=y_val, y_pred=val_preds_lr,
                   label_names=label_names,
                   classes=lr.model.classes_,       # fixed order: same 100 classes in every table
                   texts=proc_val["text"])           # original clause text, to read mistakes

In [ ]:
# 1. per-class scores - worst F1 first (check the support column: are the worst ones small classes?)
ea.report_df().head(15)

In [ ]:
# 2. TP / FP / FN / TN per clause type - sort by FN = types the model misses most
ea.counts().sort_values("FN", ascending=False).head(15)

In [ ]:
# 3. top confused pairs: one row per (true, predicted) pair, most common first
ea.top_confusions(n=15)

In [ ]:
# 4. one weak class in detail - change the name to any class from step 1
ea.class_confusions("Assigns", n=5)

In [ ]:
# 5. read the actual mistakes
mistakes = ea.mistakes()
pct_mistakes = ((len(mistakes))/(len(y_val))*100)
print(len(mistakes), "wrong out of", len(y_val), "pct mistakes", pct_mistakes)
mistakes.head(20)

In [ ]:
# 6. confusion matrix picture (quick overview only - use step 3 for the details)
ea.plot_confusion_matrix(size=30)

CHANGED FROM (1,2) - NGRAMS (uni and bi) to (1,3) - NGRAMS (uni, bi, and tri)
    - this decreased the error between waivers and no waivers classes
    - no change in applicable law and governing law - this shows that there is a different discrepency between them that the model misses

### 7b. Complement Naive Bayes

In [ ]:
ea_nb = ErrorAnalysis(y_true=y_val, y_pred=val_preds_nb,
                      label_names=label_names,
                      classes=nb.model.classes_,
                      texts=proc_val["text"])

display([ea_nb.report_df().head(15),     # worst clause types
         ea_nb.top_confusions(n=15)])     # most common mix-ups

### 7c. Logistic regression vs Naive Bayes, per clause type
Does Complement NB do better on any clause types (especially small ones)?

In [ ]:
per_class = pd.DataFrame({
    "support": ea.report_df()["support"],
    "LR F1": ea.report_df()["f1-score"],
    "NB F1": ea_nb.report_df()["f1-score"],
})
per_class["NB - LR"] = per_class["NB F1"] - per_class["LR F1"]

print("clause types where NB beats LR:", (per_class["NB - LR"] > 0).sum())
per_class.sort_values("NB - LR", ascending=False).head(15).round(3)   # NB's biggest wins

### 7d. Error Analysis Part 2: applicable law and governing law

In [ ]:
train = pd.read_parquet("../data/interim/train_eda.parquet")
print("loaded train:", train.shape)   # expect (59933, 5)
overlapping = train[train['label_name'].isin(['Applicable Laws', 'Governing Laws'])]
# they are not labelled differently with the same text == no duplicates 
overlapping["text"].duplicated().any()

Check modelling coefficients - looking at the weights;

In [ ]:
# name -> label number (flip your label_names into a lookup)
# flip the label names fictionary around - the key name and then the numerical value encoded
name_to_label = {v: k for k, v in label_names.items()}
classes = list(lr.model.classes_) # 0-99

# feature_names - column headers
# classes - rows 
# values. inside the cells = coefficient 
def class_weights(name):
    row = classes.index(name_to_label[name]) # which row of coef_ co-ordinates the list of model.classes intot he labelled names
    return pd.Series(lr.model.coef_[row], index=feature_names)

gl = class_weights("Governing Laws")
al = class_weights("Applicable Laws")

# 1. top 20 words for each class, side by side
display(pd.DataFrame({
    "Governing Laws": gl.sort_values(ascending=False).head(20).index,
    "Applicable Laws": al.sort_values(ascending=False).head(20).index,
}))

# 2. what separates them: positive = pushes to GL, negative = pushes to AL
diff = (gl - al).sort_values()
print(diff.tail(20))   # most "Governing Laws" words
print(diff.head(20))   # most "Applicable Laws" words

The more positive values: show that the result is governing law, more negative: applicable law:
- count what % of applicable law contains govern; what % of governing contains applicable law, if both are high , an overlap is evident

In [ ]:
gov_df = proc_train[proc_train['label'].isin([47])]
app_df = proc_train[proc_train['label'].isin([4])]
app_df[app_df['clean_text'].str.contains(r'\bgovern(?:s|ed|ing)?\b')] # 258
gov_df[gov_df['clean_text'].str.contains('applicable law', case = False)] # 113
len(app_df) # 349
len(gov_df)  #3157

About three-quarters of Applicable Laws clauses are choice-of-law clauses: the same kind of clause as Governing Laws, filed under a different heading.

74% of Applicable Laws clauses are choice-of-law clauses with the same wording as Governing Laws. The labels come from drafters’ headings, so the two classes overlap.

## 8. Save the trained models
So a restart doesn't mean retraining - reload with `joblib.load(...)` and go straight to evaluating.

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(lr.model, models_dir / "logreg_C30_balanced.joblib")
joblib.dump(nb.model, models_dir / "complement_nb_alpha0.1.joblib")

# to reload later (no retraining):
# lr_model = joblib.load(models_dir / "logreg_C30_balanced.joblib")
# nb_model = joblib.load(models_dir / "complement_nb_alpha0.1.joblib")

## 9. Test (ONCE, at the very end)
Settings are final - no changes after this. Both final models are scored once, for the results table.

In [ ]:
test_preds_lr, test_scores_lr = ev.eval_test(lr.model)
test_preds_nb, test_scores_nb = ev.eval_test(nb.model)

results = pd.DataFrame({
    "Logistic regression": {"val macro F1": scores_lr["validation"]["macro_f1"],
                            "test macro F1": test_scores_lr["macro_f1"],
                            "test accuracy": test_scores_lr["accuracy"]},
    "Complement NB": {"val macro F1": scores_nb["validation"]["macro_f1"],
                      "test macro F1": test_scores_nb["macro_f1"],
                      "test accuracy": test_scores_nb["accuracy"]},
}).T.round(4)
results

In [ ]:
ea_test = ErrorAnalysis(y_true=y_test, y_pred=test_preds_lr,
                   label_names=label_names,
                   classes=lr.model.classes_,       # fixed order: same 100 classes in every table
                   texts=proc_test["text"])           # original clause text, to read mistakes

In [ ]:
display([ea_test.counts().sort_values("FN", ascending=False).head(15),
         ea_test.top_confusions(n=15),
         ea_test.class_confusions("Assigns", n=5)])

In [ ]:
mistakes_test = ea_test.mistakes()
pct_mistakes_test = ((len(mistakes_test))/(len(y_test))*100)
print(len(mistakes_test), "wrong out of", len(y_test), "pct mistakes", pct_mistakes_test)

mistakes_test.head(20)
ea_test.plot_confusion_matrix(size=30)